In [9]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input
from tensorflow.keras.layers import ConvLSTM1D, Flatten, Dense

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

class WeightLogger(tf.keras.callbacks.Callback):
    def __init__(self, layer_index=0):
        self.layer_index = layer_index
        self.weights_per_epoch = []

    def on_epoch_end(self, epoch, logs=None):
        weights = self.model.layers[self.layer_index].get_weights()[0]
        self.weights_per_epoch.append(weights.copy())


def load_and_prepare_data(csv_path, production_column='production', window_size=21):
    df = pd.read_csv(csv_path, sep=',')
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(df.values)
    target_scaler = MinMaxScaler()
    target_scaler.fit(df[[production_column]])
    target_col_idx = df.columns.get_loc(production_column)
    x, y = [], []
    for i in range(window_size, len(data_scaled)):
        x.append(data_scaled[i-window_size:i])
        y.append(data_scaled[i, target_col_idx])
    x, y = np.array(x), np.array(y)
    train_split_index = int(0.8 * len(x))
    test_split_index = int(0.9 * len(x))
    x_train, y_train = x[:train_split_index], y[:train_split_index]
    x_test, y_test = x[train_split_index:test_split_index], y[train_split_index:test_split_index]
    x_val, y_val = x[test_split_index:], y[test_split_index:]
    x_train_conv = np.expand_dims(x_train, axis=2)
    x_test_conv = np.expand_dims(x_test, axis=2)
    x_val_conv = np.expand_dims(x_val, axis=2)
    return x_train_conv, y_train, x_test_conv, y_test, x_val_conv, y_val, df, target_scaler


def build_convlstm_model(lr, filters1, filters2, dense_units, input_shape):
    model = Sequential([
        ConvLSTM1D(filters=filters1, kernel_size=(1,), activation='tanh',
                   return_sequences=True, input_shape=input_shape),
        ConvLSTM1D(filters=filters2, kernel_size=(1,), activation='tanh', return_sequences=False),
        Flatten(),
        Dense(units=dense_units, activation='relu'),
        Dense(1, activation="linear")
    ])
    optimizer = Adam(learning_rate=lr)
    model.compile(loss="mae", optimizer=optimizer)
    return model


def train_and_evaluate_model(model, x_train, y_train, x_val, y_val,
                             epochs=20, batch_size=512, verbose=1, dataset_name="dataset"):
    stop_early = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    weight_logger = WeightLogger(layer_index=0)
    start_time = time.time()
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=epochs,
                        batch_size=batch_size,
                        verbose=verbose,
                        callbacks=[stop_early, weight_logger])
    training_time = time.time() - start_time
    return history, training_time, weight_logger


def inference_and_plot(model, x_test, y_test, target_scaler, dataset_name): 
    preds = model.predict(x_test)
    y_test_real = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_real = target_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test_real, y_pred_real)
    mse = mean_squared_error(y_test_real, y_pred_real)
    r2 = r2_score(y_test_real, y_pred_real)

    plt.figure(figsize=(10, 6))
    plt.plot(y_test_real, label="Vrai")
    plt.plot(y_pred_real, label="Prévu")
    plt.legend()
    plt.title(f"{dataset_name} - MAE: {mae:.4f} | R²: {r2:.4f} | MSE: {mse:.4f}")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_courbe_perffixed.png")
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test_real, y_pred_real, alpha=0.7, color='orange')
    plt.plot([min(y_test_real), max(y_test_real)], [min(y_test_real), max(y_test_real)], 'r--')
    plt.xlabel("Réel")
    plt.ylabel("Prédit")
    plt.title("Scatter plot")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_scatter_perffixed.png")
    plt.close()

    errors = y_test_real - y_pred_real
    plt.figure(figsize=(8, 4))
    plt.hist(errors, bins=30, color='orange', edgecolor='black')
    plt.title("Histogramme des erreurs")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_hist_errorsfixed.png")
    plt.close()

    safe_y_test_real = np.where(y_test_real == 0, np.nan, y_test_real)
    df_stats = pd.DataFrame({
        "Y_test": y_test_real,
        "Y_pred": y_pred_real,
        "Error": errors,
        "Error_Percent": np.abs(errors) / safe_y_test_real * 100
    })
    df_stats.describe().to_csv(f"{dataset_name}_stats_erreursfixed.csv")
    return mae, mse, r2


def run_fixed_model(csv_path, dataset_name,
                    lr=0.001,
                    filters1=64,
                    filters2=32,
                    dense_units=64,
                    epochs=200,
                    batch_size=256):
    
    x_train, y_train, x_test, y_test, x_val, y_val, df, target_scaler = load_and_prepare_data(csv_path)
    
    # === 🔽 SAUVEGARDE DU SCALER ===
    joblib.dump(target_scaler, "fixed_convlstm_target_scaler.save")
    print("Scaler sauvegardé dans fixed_convlstm_target_scaler.save")
    
    
    input_shape = x_train.shape[1:]

    model = build_convlstm_model(lr, filters1, filters2, dense_units, input_shape)
    history, training_time, _ = train_and_evaluate_model(model, x_train, y_train, x_val, y_val,
                                                         epochs=epochs, batch_size=batch_size, dataset_name=dataset_name)
    mae, mse, r2 = inference_and_plot(model, x_test, y_test, target_scaler, dataset_name)
    model.save("fixed_convlstm_model.h5")
    print(f"{dataset_name} — MAE: {mae:.4f}, MSE: {mse:.4f}, R²: {r2:.4f}")


if __name__ == "__main__":
    # 🔧 Renseigne ici les hyperparamètres souhaités
    run_fixed_model(
        csv_path="../../DataCleaning/scaled_dataset.csv",
        dataset_name="scaled_dataset_fixed",
        lr=0.00725,
        filters1=65,
        filters2=118,
        dense_units=115,
        epochs=200,
        batch_size=256
    )


Scaler sauvegardé dans fixed_convlstm_target_scaler.save
Epoch 1/200


c:\Users\Natha\anaconda3\envs\IR_pv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 101ms/step - loss: 0.2048 - val_loss: 0.0294
Epoch 2/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - loss: 0.0415 - val_loss: 0.0272
Epoch 3/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 92ms/step - loss: 0.0390 - val_loss: 0.0275
Epoch 4/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 0.0396 - val_loss: 0.0264
Epoch 5/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - loss: 0.0373 - val_loss: 0.0274
Epoch 6/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 135ms/step - loss: 0.0370 - val_loss: 0.0263
Epoch 7/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 136ms/step - loss: 0.0363 - val_loss: 0.0249
Epoch 8/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - loss: 0.0360 - val_loss: 0.0243
Epoch 9/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 142ms/step - loss: 0.0360 - val_loss: 0.0240
Epoch 10/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 167ms/step - loss: 0.0344 - val_loss: 0.0255
Epoch 11/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 153ms/step - loss: 0.0362 - val_loss: 0.0234
Epoch 12/200
34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 154ms/step -

scaled_dataset_fixed — MAE: 2.2024, MSE: 14.8233, R²: 0.9541
